In [1]:
# Cell 1: Setup

import tensorflow as tf
from tensorflow import keras                # type: ignore
from tensorflow.keras import layers         # type: ignore
import numpy as np
import matplotlib.pyplot as plt

# GPU configuration for better performance and stability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.config.optimizer.set_jit(True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

np.random.seed(42)
tf.random.set_seed(42)

2026-05-01 02:44:13.533460: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 02:44:13.592903: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-01 02:44:15.162991: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Cell 2: Data (CIFAR-10)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(50000).batch(128).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.batch(128).prefetch(tf.data.AUTOTUNE)

y_train = y_train.squeeze()
y_test = y_test.squeeze()

/home/username/tf-env/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")
I0000 00:00:1777599858.903657  831152 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [ ]:
# Cell 3: Conv Hebbian Layer

class HebbConv2D(layers.Layer):
    def __init__(self, filters, kernel_size=3, lr=0.001, **kwargs):
        super().__init__(**kwargs)

        self.filters = filters
        self.kernel_size = kernel_size
        self.lr = lr

    def build(self, input_shape):

        in_channels = input_shape[-1]

        self.w = self.add_weight(
            shape=(self.kernel_size, self.kernel_size, in_channels, self.filters),
            initializer="glorot_uniform",
            trainable=False,   # IMPORTANT: Hebbian, not gradient descent
            name="hebb_weight"
        )

    def call(self, x, training=False):

        # Forward pass
        y = tf.nn.conv2d(x, self.w, strides=1, padding="SAME")

        if training:

            patches = tf.image.extract_patches(
                images=x,
                sizes=[1, self.kernel_size, self.kernel_size, 1],
                strides=[1, 1, 1, 1],
                rates=[1, 1, 1, 1],
                padding="SAME"
            )

            pre = tf.reduce_mean(patches, axis=[0, 1, 2]) 

            post = tf.reduce_mean(y, axis=[0, 1, 2])

            hebb = tf.tensordot(pre, post, axes=0)

            hebb = tf.reshape(
                hebb,
                (self.kernel_size, self.kernel_size, x.shape[-1], self.filters)
            )

            # Oja stabilisation
            norm = tf.reduce_mean(post)
            decay = self.w * norm

            dw = hebb - decay

            self.w.assign_add(self.lr * tf.cast(dw, self.w.dtype))

        return y

In [4]:
# Cell 4: CNN

def build_hebb_net():

    inputs = keras.Input(shape=(32, 32, 3))

    x = HebbConv2D(32)(inputs)
    x = layers.ReLU()(x)

    x = layers.MaxPool2D()(x)

    x = HebbConv2D(64)(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    return keras.Model(inputs, outputs)

model = build_hebb_net()

In [ ]:
# Cell 5: Training loop (Hebbian learning)

loss_fn = keras.losses.SparseCategoricalCrossentropy()

optimizer = keras.optimizers.Adam(1e-3)

for epoch in range(5):

    print("\nEpoch", epoch+1)

    for step, (x, y) in enumerate(train_ds):

        with tf.GradientTape() as tape:

            logits = model(x, training=True)
            loss = loss_fn(y, logits)

        # ONLY train readout layer
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if step % 100 == 0:
            print("step", step, "loss", float(loss))


Epoch 1


2026-05-01 02:44:22.163512: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


step 0 loss 2.2932729721069336
step 100 loss 2.2860939502716064
step 200 loss 2.305354118347168
step 300 loss 67.39295959472656


2026-05-01 02:44:30.322191: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 2
step 0 loss nan
step 100 loss nan
step 200 loss nan
step 300 loss nan


2026-05-01 02:44:39.330128: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 3
step 0 loss nan
step 100 loss nan
step 200 loss nan
step 300 loss nan

Epoch 4
step 0 loss nan
step 100 loss nan
step 200 loss nan
step 300 loss nan


2026-05-01 02:44:54.914091: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 5
step 0 loss nan
step 100 loss nan
step 200 loss nan
step 300 loss nan
